# Question 1 — reproducible panel preparation

This notebook is the single preparation pipeline for Question 1. It reads the eight respondent
files once, applies the official survey weights, joins only defensible external context, validates
every output and writes six analysis-ready panels.

**Weighting contract**

- annual estimates use `wt_final`;
- monthly estimates use `wt_time`;
- the 372 activity fields and three survey-composition fields are preserved;
- external variables are joined after survey aggregation and are never survey-weighted;
- no train/validation/test label is stored upstream.

Actual ONS population and migration, residence-based earnings, APS economic inactivity and
observed historical weather are included. Projections, incomplete or reclassified crime data and
empty Nomis downloads are documented but excluded. Forecasting notebooks must lag all eligible
predictors themselves.


In [ ]:
from pathlib import Path
import csv
import json
import os
import re

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'physical_activity_analysis' else cwd
OUTPUT_DIR = PROJECT_ROOT / 'physical_activity_analysis' / 'csv_outputs'
PROJECT_EXTERNAL_DIR = PROJECT_ROOT / 'physical_activity_analysis' / 'external_data'
EXTRA_DATA_DIR = Path(
    os.getenv('LONDON_SPORT_EXTRA_DATA_DIR', PROJECT_ROOT.parent / 'extradata')
).resolve()
WAVES_PATH = PROJECT_ROOT / 'data_integration' / 'waves.json'
LOOKUP_PATH = (
    PROJECT_ROOT / 'Jingyi Hua' / 'data' / 'processed' / 'variable_value_labels_lookup_year7_8.csv'
)

EXTERNAL_PATHS = {
    'ons_population_migration': EXTRA_DATA_DIR / 'myebtablesenglandwales20112024.xlsx',
    'resident_earnings': EXTRA_DATA_DIR / 'earnings-residence-borough.xlsx',
    'weather_2017_2023': EXTRA_DATA_DIR / 'open-meteo-51.49N0.16W23m.csv',
    'economic_inactivity': PROJECT_EXTERNAL_DIR / 'economic-inactivity.csv',
    'weather_2015_2016': PROJECT_EXTERNAL_DIR / 'open-meteo-2015-2016.csv',
}

OUTPUTS = {
    'london_annual': OUTPUT_DIR / 'question1_london_annual.csv',
    'area_annual': OUTPUT_DIR / 'question1_inner_outer_annual.csv',
    'borough_annual': OUTPUT_DIR / 'question1_borough_annual.csv',
    'london_monthly': OUTPUT_DIR / 'question1_london_monthly.csv',
    'area_monthly': OUTPUT_DIR / 'question1_inner_outer_monthly.csv',
    'borough_monthly': OUTPUT_DIR / 'question1_borough_monthly.csv',
}
RAW_MISSINGNESS_DETAIL_OUTPUT = OUTPUT_DIR / 'question1_raw_missingness_by_wave_variable.csv'
RAW_MISSINGNESS_SUMMARY_OUTPUT = OUTPUT_DIR / 'question1_raw_missingness_variable_summary.csv'
EXTERNAL_CONTEXT_OUTPUT = OUTPUT_DIR / 'question1_external_context_borough_year.csv'
EXTERNAL_INVENTORY_OUTPUT = OUTPUT_DIR / 'question1_external_data_inventory.csv'
EXTERNAL_QUALITY_OUTPUT = OUTPUT_DIR / 'question1_external_quality_summary.csv'
VARIABLE_DICTIONARY_OUTPUT = OUTPUT_DIR / 'question1_variable_dictionary.csv'

assert WAVES_PATH.exists() and LOOKUP_PATH.exists()
assert all(path.exists() for path in EXTERNAL_PATHS.values()), EXTERNAL_PATHS
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SPSS_MISSING = list(range(-99, -89))
CITY_OF_LONDON_LA2023 = 59
AREA_LABELS = {1: 'Inner London', 2: 'Outer London'}
AREA_CODES = {1: 'E13000001', 2: 'E13000002'}
FIRST_SURVEY_PERIOD = pd.Period('2015-11', freq='M')
COMPOSITION_FEATURES = ['older_adult_rate', 'limiting_disability_rate', 'online_response_rate']
OUTCOME_COLUMNS = ['inactive_rate', 'fairly_active_rate', 'active_rate']
QUALITY_COLUMNS = ['effective_n']

SOCIODEMOGRAPHIC_CONTEXT_COLUMNS = [
    'adult_population',
    'population_age_16_34_rate',
    'population_age_65_plus_rate',
    'population_female_rate',
    'net_migration_per_1000',
    'median_weekly_earnings_gbp',
    'earnings_relative_to_london',
    'economic_inactivity_rate',
]
EXTERNAL_QUALITY_COLUMNS = ['earnings_confidence_pct', 'economic_inactivity_confidence_pp']
WEATHER_CONTEXT_COLUMNS = [
    'weather_mean_temperature_c',
    'weather_total_precipitation_mm',
    'weather_total_sunshine_hours',
    'weather_mean_max_wind_kmh',
    'weather_wet_day_rate',
]
EXTERNAL_PANEL_COLUMNS = (
    ['context_reference_year']
    + SOCIODEMOGRAPHIC_CONTEXT_COLUMNS
    + EXTERNAL_QUALITY_COLUMNS
    + WEATHER_CONTEXT_COLUMNS
)

waves = json.loads(WAVES_PATH.read_text(encoding='utf-8'))
for wave in waves:
    wave['path'] = PROJECT_ROOT / wave['relative_path']
    assert wave['path'].exists(), wave['path']
assert [wave['wave_index'] for wave in waves] == list(range(1, 9))


## 1. External context: source decisions and harmonisation

**Purpose.** Build a tidy `borough × calendar year` context table and two observed-weather tables
without mixing projections or incomplete series into the analysis.

**Method.** Annual survey waves use their ending calendar year (2016–2023); monthly rows use the
calendar year of the observation. Counts are retained for defensible aggregation. Broader-area
earnings are population-weighted summaries of borough medians—not individual-level medians.


In [ ]:
external_inventory = pd.DataFrame(
    [
        {
            'file': 'myebtablesenglandwales20112024.xlsx',
            'decision': 'included',
            'role': 'actual population age/sex structure and net migration',
            'reason': 'official annual estimates; complete 2015–2023 borough coverage',
            'source': 'ONS mid-year population estimates and components of change',
        },
        {
            'file': 'earnings-residence-borough.xlsx',
            'decision': 'included',
            'role': 'residence-based median weekly earnings',
            'reason': 'complete 2015–2023 coverage for the 32 study boroughs',
            'source': 'ONS Annual Survey of Hours and Earnings via London Datastore',
        },
        {
            'file': 'economic-inactivity.csv',
            'decision': 'included',
            'role': 'working-age economic inactivity rate',
            'reason': 'official APS annual series with complete study coverage',
            'source': 'ONS Annual Population Survey via London Datastore',
        },
        {
            'file': 'open-meteo historical weather data',
            'decision': 'included',
            'role': 'London-wide monthly weather context',
            'reason': 'daily historical values cover every month from 2015-11 to 2023-10',
            'source': 'Open-Meteo historical weather API and user-supplied extract',
        },
        {
            'file': '2022_Identified_Capacity_10yr_central_fert_std.xlsx',
            'decision': 'excluded',
            'role': 'GLA demographic projections',
            'reason': 'projections are unnecessary because actual ONS estimates cover the study',
            'source': 'GLA 2022-based demographic projections',
        },
        {
            'file': 'open-meteo-51.50N0.10W23m.csv',
            'decision': 'excluded',
            'role': '2024–2030 climate-model projections',
            'reason': 'future model scenarios are not observations and do not cover study outcomes',
            'source': 'Open-Meteo climate projections',
        },
        {
            'file': 'MPS Borough Level Crime.csv',
            'decision': 'excluded',
            'role': 'borough crime',
            'reason': 'starts 2020-06 and offence definitions change inside the short series',
            'source': 'Metropolitan Police Service',
        },
        {
            'file': 'nomis_2026_07_14_*.xlsx',
            'decision': 'excluded',
            'role': 'intended APS extracts',
            'reason': 'both workbooks contain only a title and no observations',
            'source': 'Nomis download attempts',
        },
    ]
)
external_inventory.to_csv(EXTERNAL_INVENTORY_OUTPUT, index=False, encoding='utf-8-sig')
display(external_inventory)


### Population and migration

Actual ONS estimates provide denominators, age structure, sex structure and net migration for all 32 boroughs and nine calendar years.


In [ ]:
# ONS population by sex and single year of age.
population_raw = pd.read_excel(
    EXTERNAL_PATHS['ons_population_migration'], sheet_name='MYEB1', header=1
)
population_raw = population_raw.loc[
    population_raw['ladcode23'].astype(str).str.startswith('E090')
    & population_raw['ladcode23'].ne('E09000001')
].copy()
population_raw['age'] = pd.to_numeric(population_raw['age'], errors='coerce')
population_columns = [f'population_{year}' for year in range(2015, 2024)]
population_long = population_raw.melt(
    id_vars=['ladcode23', 'laname23', 'sex', 'age'],
    value_vars=population_columns,
    var_name='calendar_year',
    value_name='population',
)
population_long['calendar_year'] = (
    population_long['calendar_year'].str.extract(r'(\d{4})').astype(int)
)
population_long['population'] = pd.to_numeric(population_long['population'], errors='coerce')
assert population_long['population'].notna().all()

population_context_rows = []
for (code_value, name_value, calendar_year), group in population_long.groupby(
    ['ladcode23', 'laname23', 'calendar_year'], sort=True
):
    adults = group.loc[group['age'].ge(16)]
    adult_population = adults['population'].sum()
    population_context_rows.append(
        {
            'geography_code': code_value,
            'geography_name': name_value,
            'calendar_year': calendar_year,
            'total_population': group['population'].sum(),
            'adult_population': adult_population,
            'age_16_34_population': adults.loc[adults['age'].between(16, 34), 'population'].sum(),
            'age_65_plus_population': adults.loc[adults['age'].ge(65), 'population'].sum(),
            'female_adult_population': adults.loc[adults['sex'].eq('f'), 'population'].sum(),
        }
    )
population_context = pd.DataFrame(population_context_rows)

# ONS migration components, expressed per 1,000 total residents.
migration_raw = pd.read_excel(
    EXTERNAL_PATHS['ons_population_migration'], sheet_name='MYEB3', header=1
)
migration_raw = migration_raw.loc[
    migration_raw['ladcode23'].astype(str).str.startswith('E090')
    & migration_raw['ladcode23'].ne('E09000001')
].copy()
migration_rows = []
for _, row in migration_raw.iterrows():
    for calendar_year in range(2015, 2024):
        internal = pd.to_numeric(row[f'internal_net_{calendar_year}'], errors='coerce')
        international = pd.to_numeric(row[f'international_net_{calendar_year}'], errors='coerce')
        migration_rows.append(
            {
                'geography_code': row['ladcode23'],
                'calendar_year': calendar_year,
                'net_migration': internal + international,
            }
        )
migration_context = pd.DataFrame(migration_rows)


### Earnings and labour-market context

ASHE residence-based median weekly pay and APS economic inactivity are parsed with their published quality fields before the sources are joined.


In [ ]:
# Residence-based ASHE total median weekly pay and its published confidence percentage.
earnings_sheet = pd.read_excel(
    EXTERNAL_PATHS['resident_earnings'], sheet_name='Total, weekly', header=None
)
earnings_rows = []
for column_index in range(2, earnings_sheet.shape[1], 2):
    calendar_year = pd.to_numeric(earnings_sheet.iloc[0, column_index], errors='coerce')
    if pd.isna(calendar_year) or not 2015 <= int(calendar_year) <= 2023:
        continue
    for row_index in range(3, earnings_sheet.shape[0]):
        area_name = earnings_sheet.iloc[row_index, 1]
        earnings_rows.append(
            {
                'geography_name': area_name,
                'calendar_year': int(calendar_year),
                'median_weekly_earnings_gbp': pd.to_numeric(
                    earnings_sheet.iloc[row_index, column_index], errors='coerce'
                ),
                'earnings_confidence_pct': pd.to_numeric(
                    earnings_sheet.iloc[row_index, column_index + 1], errors='coerce'
                ),
            }
        )
earnings_context = pd.DataFrame(earnings_rows).dropna(subset=['geography_name'])

# APS economic inactivity. Counts allow exact aggregation above borough level.
inactivity_raw = pd.read_csv(EXTERNAL_PATHS['economic_inactivity'], dtype=str)
inactivity_rows = []
for rate_column in [column for column in inactivity_raw if column.startswith('percent;')]:
    calendar_year = int(re.search(r'(20\d{2})', rate_column).group(1))
    if not 2015 <= calendar_year <= 2023:
        continue
    suffix = rate_column.split(';', 1)[1]
    working_column = f'Working age;{suffix}'
    inactive_column = f'Economically Inactive;{suffix}'
    confidence_column = f'confidence;{suffix}'
    for _, row in inactivity_raw.iterrows():
        inactivity_rows.append(
            {
                'geography_code': row['Code'],
                'calendar_year': calendar_year,
                'working_age_population': pd.to_numeric(
                    str(row[working_column]).replace(',', ''), errors='coerce'
                ),
                'economically_inactive_count': pd.to_numeric(
                    str(row[inactive_column]).replace(',', ''), errors='coerce'
                ),
                'economic_inactivity_rate': pd.to_numeric(row[rate_column], errors='coerce') / 100,
                'economic_inactivity_confidence_pp': pd.to_numeric(
                    row[confidence_column], errors='coerce'
                ),
            }
        )
inactivity_context = pd.DataFrame(inactivity_rows)

borough_external_context = (
    population_context.merge(
        migration_context, on=['geography_code', 'calendar_year'], validate='one_to_one'
    )
    .merge(earnings_context, on=['geography_name', 'calendar_year'], validate='one_to_one')
    .merge(inactivity_context, on=['geography_code', 'calendar_year'], validate='one_to_one')
)
borough_external_context['population_age_16_34_rate'] = (
    borough_external_context['age_16_34_population'] / borough_external_context['adult_population']
)
borough_external_context['population_age_65_plus_rate'] = (
    borough_external_context['age_65_plus_population']
    / borough_external_context['adult_population']
)
borough_external_context['population_female_rate'] = (
    borough_external_context['female_adult_population']
    / borough_external_context['adult_population']
)
borough_external_context['net_migration_per_1000'] = (
    1000 * borough_external_context['net_migration'] / borough_external_context['total_population']
)
london_earnings = borough_external_context.groupby('calendar_year').apply(
    lambda group: np.average(
        group['median_weekly_earnings_gbp'], weights=group['adult_population']
    ),
    include_groups=False,
)
borough_external_context['earnings_relative_to_london'] = borough_external_context[
    'median_weekly_earnings_gbp'
] / borough_external_context['calendar_year'].map(london_earnings)

assert len(borough_external_context) == 32 * 9
assert borough_external_context[['geography_code', 'calendar_year']].duplicated().sum() == 0
assert (
    borough_external_context[SOCIODEMOGRAPHIC_CONTEXT_COLUMNS + EXTERNAL_QUALITY_COLUMNS]
    .notna()
    .all()
    .all()
)


### Aggregation rules

Counts are summed; rates are rebuilt from denominators; borough medians are summarised with adult-population weights.


In [ ]:
def aggregate_borough_context(frame, group_columns):
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else group_columns
    for keys, group in frame.groupby(grouper, sort=True, observed=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        adult_total = group['adult_population'].sum()
        total_population = group['total_population'].sum()
        working_total = group['working_age_population'].sum()
        row = dict(zip(group_columns, keys))
        row.update(
            {
                'adult_population': adult_total,
                'population_age_16_34_rate': group['age_16_34_population'].sum() / adult_total,
                'population_age_65_plus_rate': group['age_65_plus_population'].sum() / adult_total,
                'population_female_rate': group['female_adult_population'].sum() / adult_total,
                'net_migration_per_1000': 1000 * group['net_migration'].sum() / total_population,
                'median_weekly_earnings_gbp': np.average(
                    group['median_weekly_earnings_gbp'], weights=group['adult_population']
                ),
                'earnings_relative_to_london': np.average(
                    group['earnings_relative_to_london'], weights=group['adult_population']
                ),
                'economic_inactivity_rate': group['economically_inactive_count'].sum()
                / working_total,
                'earnings_confidence_pct': np.average(
                    group['earnings_confidence_pct'], weights=group['adult_population']
                ),
                'economic_inactivity_confidence_pp': np.average(
                    group['economic_inactivity_confidence_pp'],
                    weights=group['working_age_population'],
                ),
            }
        )
        rows.append(row)
    return pd.DataFrame(rows)


### Observed weather

Daily London observations are reduced to the exact 96 survey months and eight November–October survey years.


In [ ]:
# Daily observed weather. The downloaded supplement closes the 2015–2016 gap.
weather_parts = [
    pd.read_csv(EXTERNAL_PATHS['weather_2015_2016'], skiprows=2),
    pd.read_csv(EXTERNAL_PATHS['weather_2017_2023'], skiprows=2),
]
weather_daily = pd.concat(weather_parts, ignore_index=True)
weather_daily['time'] = pd.to_datetime(weather_daily['time'])
weather_daily = weather_daily.drop_duplicates('time').sort_values('time')
weather_daily = weather_daily.loc[weather_daily['time'].between('2015-11-01', '2023-10-31')].copy()


def weather_column(prefix):
    return next(column for column in weather_daily.columns if column.startswith(prefix))


temperature_max = weather_column('temperature_2m_max')
temperature_min = weather_column('temperature_2m_min')
precipitation = weather_column('precipitation_sum')
sunshine = weather_column('sunshine_duration')
wind = weather_column('wind_speed_10m_max')
weather_daily['mean_temperature'] = (
    pd.to_numeric(weather_daily[temperature_max], errors='coerce')
    + pd.to_numeric(weather_daily[temperature_min], errors='coerce')
) / 2
weather_daily['precipitation'] = pd.to_numeric(weather_daily[precipitation], errors='coerce')
weather_daily['sunshine_hours'] = pd.to_numeric(weather_daily[sunshine], errors='coerce') / 3600
weather_daily['max_wind'] = pd.to_numeric(weather_daily[wind], errors='coerce')
weather_daily['wet_day'] = weather_daily['precipitation'].ge(1).astype(float)
assert (
    weather_daily[['mean_temperature', 'precipitation', 'sunshine_hours', 'max_wind']]
    .notna()
    .all()
    .all()
)

weather_daily['period_start'] = weather_daily['time'].dt.to_period('M').astype(str)
weather_monthly_context = weather_daily.groupby('period_start', as_index=False).agg(
    weather_mean_temperature_c=('mean_temperature', 'mean'),
    weather_total_precipitation_mm=('precipitation', 'sum'),
    weather_total_sunshine_hours=('sunshine_hours', 'sum'),
    weather_mean_max_wind_kmh=('max_wind', 'mean'),
    weather_wet_day_rate=('wet_day', 'mean'),
)
weather_daily['month_index'] = (
    weather_daily['time'].dt.to_period('M') - FIRST_SURVEY_PERIOD
).apply(lambda offset: offset.n) + 1
weather_daily['year'] = ((weather_daily['month_index'] - 1) // 12 + 1).astype(int)
weather_annual_context = weather_daily.groupby('year', as_index=False).agg(
    weather_mean_temperature_c=('mean_temperature', 'mean'),
    weather_total_precipitation_mm=('precipitation', 'sum'),
    weather_total_sunshine_hours=('sunshine_hours', 'sum'),
    weather_mean_max_wind_kmh=('max_wind', 'mean'),
    weather_wet_day_rate=('wet_day', 'mean'),
)
assert len(weather_monthly_context) == 96 and len(weather_annual_context) == 8
assert weather_monthly_context[WEATHER_CONTEXT_COLUMNS].notna().all().all()


### External-data quality audit and export

The final block checks completeness and ranges, then writes the source register, borough-year context and variable-level quality summary.


In [ ]:
context_export_columns = (
    ['calendar_year', 'geography_code', 'geography_name']
    + SOCIODEMOGRAPHIC_CONTEXT_COLUMNS
    + EXTERNAL_QUALITY_COLUMNS
)
borough_external_context[context_export_columns].to_csv(
    EXTERNAL_CONTEXT_OUTPUT, index=False, encoding='utf-8-sig'
)
external_quality_summary = pd.DataFrame(
    [
        {
            'variable': column,
            'rows': len(borough_external_context),
            'missing': int(borough_external_context[column].isna().sum()),
            'minimum': borough_external_context[column].min(),
            'median': borough_external_context[column].median(),
            'maximum': borough_external_context[column].max(),
        }
        for column in SOCIODEMOGRAPHIC_CONTEXT_COLUMNS + EXTERNAL_QUALITY_COLUMNS
    ]
)
external_quality_summary.to_csv(EXTERNAL_QUALITY_OUTPUT, index=False, encoding='utf-8-sig')
display(external_quality_summary)


### Findings

- Four sources are included and four are excluded with explicit reasons.
- The socioeconomic/demographic table contains **288 complete borough-years** (32 boroughs ×
  2015–2023); all ten retained context/quality variables have zero missing values.
- Observed weather covers all **96 months** and all eight survey years.
- Population projections, climate projections, the short reclassified crime series and empty
  Nomis files do not enter any prepared panel.


## 2. Stable activity schema

**Purpose.** Prevent a variable from appearing only because it exists in a subset of survey waves.

**Method.** Retain an activity only when both its 12-month and recent-participation source fields
exist in every wave. The same suffix list generates the three output families.


In [ ]:
def read_header(path):
    with open(path, encoding='utf-8-sig', newline='') as handle:
        return next(csv.reader(handle))


headers = {wave['wave_index']: read_header(wave['path']) for wave in waves}
months_sets = {
    index: {
        column.removeprefix('MONTHS_12_') for column in header if column.startswith('MONTHS_12_')
    }
    for index, header in headers.items()
}
days_sets = {
    index: {
        column.removeprefix('DAYS10P60GR_')
        for column in header
        if column.startswith('DAYS10P60GR_')
    }
    for index, header in headers.items()
}
common_suffixes = set.intersection(*months_sets.values()) & set.intersection(*days_sets.values())
ACTIVITY_SUFFIXES = [
    column.removeprefix('MONTHS_12_')
    for column in headers[1]
    if column.startswith('MONTHS_12_') and column.removeprefix('MONTHS_12_') in common_suffixes
]
MONTHS_COLUMNS = [f'MONTHS_12_{suffix}' for suffix in ACTIVITY_SUFFIXES]
DAYS_COLUMNS = [f'DAYS10P60GR_{suffix}' for suffix in ACTIVITY_SUFFIXES]
MONTHS12_RATE_COLUMNS = [f'MONTHS12_RATE_{suffix}' for suffix in ACTIVITY_SUFFIXES]
DAYS_ANY_RATE_COLUMNS = [f'DAYS_ANY_RATE_{suffix}' for suffix in ACTIVITY_SUFFIXES]
DAYS_2PLUS_RATE_COLUMNS = [f'DAYS_2PLUS_RATE_{suffix}' for suffix in ACTIVITY_SUFFIXES]
ACTIVITY_FEATURES = MONTHS12_RATE_COLUMNS + DAYS_ANY_RATE_COLUMNS + DAYS_2PLUS_RATE_COLUMNS

assert len(ACTIVITY_SUFFIXES) == 124
assert len(ACTIVITY_FEATURES) == 372 and len(set(ACTIVITY_FEATURES)) == 372
assert 'HULAHOOP_P27' not in ACTIVITY_SUFFIXES


### Findings

The strict intersection contains **124 activities**. Each produces a 12-month, any-recent and
2+-recent rate, giving **372 stable activity variables**. `HULAHOOP_P27`, which is not common to
all waves, is correctly excluded.


## 3. Raw Question 1 missingness

**Purpose.** Document missingness before eligibility filtering or weighting.

**Method.** Each wave is read in chunks. Blank values and SPSS codes `-99` to `-90` are counted
separately for the 257 fields used to construct Question 1. Routed activity missingness remains
missing; it is never converted to zero.


In [ ]:
RAW_AUDIT_CHUNK_SIZE = 5000
Q1_RAW_BASE_COLUMNS = [
    'mode',
    'LA_2023',
    'LondInOut',
    'Age9',
    'Disab3',
    'wt_final',
    'wt_time',
    'MEMS7GR_ALL',
]
SPSS_MISSING_TOKENS = list(
    {token for code in SPSS_MISSING for token in (code, float(code), str(code), f'{code}.0')}
)


def audit_raw_wave_missingness(wave):
    header = headers[wave['wave_index']]
    month_source = next(column for column in header if column.lower() == 'month')
    required = set(Q1_RAW_BASE_COLUMNS + [month_source] + MONTHS_COLUMNS + DAYS_COLUMNS)
    q1_columns = [column for column in header if column in required]
    assert len(q1_columns) == 257
    assert set(q1_columns) == required

    blank_counts = np.zeros(len(q1_columns), dtype=np.int64)
    spss_counts = np.zeros(len(q1_columns), dtype=np.int64)
    row_count = 0

    for chunk in pd.read_csv(
        wave['path'], usecols=q1_columns, chunksize=RAW_AUDIT_CHUNK_SIZE, low_memory=False
    ):
        chunk = chunk[q1_columns]
        row_count += len(chunk)
        blank_counts += chunk.isna().sum(axis=0).to_numpy(dtype=np.int64)
        spss_counts += chunk.isin(SPSS_MISSING_TOKENS).sum(axis=0).to_numpy(dtype=np.int64)

    total_missing = blank_counts + spss_counts
    assert (total_missing <= row_count).all()
    return pd.DataFrame(
        {
            'wave_index': wave['wave_index'],
            'survey_wave': wave['survey_wave'],
            'source_file': wave['path'].name,
            'variable_position': [header.index(column) + 1 for column in q1_columns],
            'variable': q1_columns,
            'rows_evaluated': row_count,
            'blank_na_count': blank_counts,
            'spss_missing_code_count': spss_counts,
            'total_missing_count': total_missing,
            'observed_count': row_count - total_missing,
            'missing_rate': total_missing / row_count,
            'has_missing': total_missing > 0,
        }
    )


### Execute and summarise the audit

The detailed wave-variable audit and the across-wave variable summary are exported separately.


In [ ]:
raw_missingness_detail = (
    pd.concat([audit_raw_wave_missingness(wave) for wave in waves], ignore_index=True)
    .sort_values(['wave_index', 'variable_position'])
    .reset_index(drop=True)
)

raw_missingness_summary = raw_missingness_detail.groupby('variable', as_index=False).agg(
    waves_present=('wave_index', 'nunique'),
    rows_evaluated=('rows_evaluated', 'sum'),
    blank_na_count=('blank_na_count', 'sum'),
    spss_missing_code_count=('spss_missing_code_count', 'sum'),
    total_missing_count=('total_missing_count', 'sum'),
    observed_count=('observed_count', 'sum'),
)
raw_missingness_summary['waves_absent'] = len(waves) - raw_missingness_summary['waves_present']
raw_missingness_summary['missing_rate'] = (
    raw_missingness_summary['total_missing_count'] / raw_missingness_summary['rows_evaluated']
)
raw_missingness_summary['has_missing'] = raw_missingness_summary['total_missing_count'].gt(0)
raw_missingness_summary = raw_missingness_summary.sort_values(
    ['missing_rate', 'total_missing_count', 'variable'], ascending=[False, False, True]
).reset_index(drop=True)

raw_missingness_detail.to_csv(RAW_MISSINGNESS_DETAIL_OUTPUT, index=False, encoding='utf-8-sig')
raw_missingness_summary.to_csv(RAW_MISSINGNESS_SUMMARY_OUTPUT, index=False, encoding='utf-8-sig')

wave_missingness_overview = raw_missingness_detail.groupby(
    ['wave_index', 'survey_wave', 'source_file'], as_index=False
).agg(
    rows=('rows_evaluated', 'first'),
    variables=('variable', 'size'),
    variables_with_missing=('has_missing', 'sum'),
    missing_cells=('total_missing_count', 'sum'),
    cells_evaluated=('rows_evaluated', 'sum'),
)
wave_missingness_overview['overall_missing_rate'] = (
    wave_missingness_overview['missing_cells'] / wave_missingness_overview['cells_evaluated']
)

display(wave_missingness_overview.style.format({'overall_missing_rate': '{:.2%}'}))
display(
    raw_missingness_summary.loc[raw_missingness_summary['has_missing']]
    .head(40)
    .style.format({'missing_rate': '{:.2%}'})
)
assert raw_missingness_detail.groupby('wave_index').size().eq(257).all()
assert set(raw_missingness_detail['variable']) <= set(
    Q1_RAW_BASE_COLUMNS + MONTHS_COLUMNS + DAYS_COLUMNS + ['Month', 'month']
)


### Findings

- The union contains **258 source names** because the month field name changes across waves;
  every individual wave still contributes exactly 257 required fields.
- **149 variables are fully observed** across all waves and **109 contain some missingness**.
- Missingness is strongly wave- and routing-dependent (about 17% in the first two waves, near
  zero in 2018/19–2019/20, and 5–10% in the final three waves), supporting the decision to retain
  `NaN` rather than reinterpret routed values as non-participation.


## 4. Respondent cleaning

**Purpose.** Create one clean respondent table per wave for both annual and monthly aggregation.

**Method.** SPSS missing codes become `NaN`; London geography and survey month are standardised;
annual and monthly respondent sets differ only in whether the corresponding official weight is
positive.


In [ ]:
BASE_COLUMNS = [
    'mode',
    'LA_2023',
    'LondInOut',
    'Age9',
    'Disab3',
    'wt_final',
    'wt_time',
    'MEMS7GR_ALL',
]


def load_clean_wave(wave):
    header = headers[wave['wave_index']]
    month_source = next(column for column in header if column.lower() == 'month')
    usecols = BASE_COLUMNS + [month_source] + MONTHS_COLUMNS + DAYS_COLUMNS
    assert not (set(usecols) - set(header))

    frame = pd.read_csv(wave['path'], usecols=usecols, low_memory=False)
    frame = frame.rename(columns={month_source: 'month_index'})
    numeric_columns = BASE_COLUMNS + ['month_index'] + MONTHS_COLUMNS + DAYS_COLUMNS
    frame[numeric_columns] = frame[numeric_columns].apply(pd.to_numeric, errors='coerce')
    frame = frame.replace(SPSS_MISSING, np.nan)

    eligible = (
        frame['MEMS7GR_ALL'].isin([0, 1, 2])
        & frame['LondInOut'].isin([1, 2])
        & frame['LA_2023'].notna()
        & frame['LA_2023'].ne(CITY_OF_LONDON_LA2023)
    )
    frame = frame.loc[
        eligible, BASE_COLUMNS + ['month_index'] + MONTHS_COLUMNS + DAYS_COLUMNS
    ].copy()
    frame.insert(0, 'survey_wave', wave['survey_wave'])
    frame.insert(0, 'year', wave['wave_index'])
    return frame


respondents = pd.concat([load_clean_wave(wave) for wave in waves], ignore_index=True)

lookup = pd.read_csv(LOOKUP_PATH, low_memory=False)
lookup['Code'] = pd.to_numeric(lookup['Code'], errors='coerce')
la_lookup = (
    lookup[lookup['year'].eq(8) & lookup['Variable'].eq('LA_2023')][['Code', 'CodeLabel']]
    .dropna()
    .drop_duplicates('Code')
)
la_lookup['gss_code'] = la_lookup['CodeLabel'].str.split().str[0]
la_lookup['borough'] = la_lookup['CodeLabel'].str.split(n=1).str[1]
respondents['borough'] = respondents['LA_2023'].map(la_lookup.set_index('Code')['borough'])
respondents['gss_code'] = respondents['LA_2023'].map(la_lookup.set_index('Code')['gss_code'])
respondents['inner_outer'] = respondents['LondInOut'].map(AREA_LABELS)
assert respondents[['borough', 'gss_code', 'inner_outer']].notna().all().all()

annual_respondents = respondents.loc[respondents['wt_final'].gt(0)].copy()
monthly_respondents = respondents.loc[respondents['wt_time'].gt(0)].copy()
assert len(annual_respondents) == 135497
assert len(monthly_respondents) == 134916
assert set(monthly_respondents['month_index'].astype(int)) == set(range(1, 97))
assert annual_respondents['borough'].nunique() == monthly_respondents['borough'].nunique() == 32


### Findings

The cleaned inputs retain **135,497 annual-weight respondents** and **134,916 monthly-weight
respondents**. The difference is expected because weight eligibility is defined separately.


## 5. Shared weighted aggregation

One reusable aggregation function calculates the three outcome shares, survey-composition fields,
372 activity rates, respondent count and Kish effective sample size. Keeping one implementation
for every geography level prevents annual/monthly definitions from drifting. External context is
joined only after this step.


In [ ]:
def kish_effective_n(weights):
    weights = np.asarray(weights, dtype=float)
    return float(weights.sum() ** 2 / np.square(weights).sum())


def weighted_binary_share(values, weights, valid_codes, positive_codes):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isin(values, valid_codes) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    return float(np.average(np.isin(values[valid], positive_codes), weights=weights[valid]))


def weighted_rate_vector(values, weights, valid_codes, positive_rule):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)[:, None]
    valid = np.isin(values, valid_codes)
    denominator = np.sum(valid * weights, axis=0)
    numerator = np.sum((positive_rule(values) & valid) * weights, axis=0)
    return np.divide(
        numerator,
        denominator,
        out=np.full(values.shape[1], np.nan, dtype=float),
        where=denominator > 0,
    )


def summarise_group(group, weight_column):
    weights = group[weight_column].to_numpy(float)
    total = weights.sum()
    row = {
        'effective_n': kish_effective_n(weights),
        'inactive_rate': float(weights[group['MEMS7GR_ALL'].eq(0).to_numpy()].sum() / total),
        'fairly_active_rate': float(weights[group['MEMS7GR_ALL'].eq(1).to_numpy()].sum() / total),
        'active_rate': float(weights[group['MEMS7GR_ALL'].eq(2).to_numpy()].sum() / total),
        'older_adult_rate': weighted_binary_share(group['Age9'], weights, range(2, 10), [7, 8, 9]),
        'limiting_disability_rate': weighted_binary_share(
            group['Disab3'], weights, [1, 2, 3], [1]
        ),
        'online_response_rate': weighted_binary_share(group['mode'], weights, [1, 2], [1]),
    }
    months12_rates = weighted_rate_vector(
        group[MONTHS_COLUMNS], weights, [0, 1], lambda values: values == 1
    )
    days_any_rates = weighted_rate_vector(
        group[DAYS_COLUMNS], weights, [0, 1, 2], lambda values: values >= 1
    )
    days_2plus_rates = weighted_rate_vector(
        group[DAYS_COLUMNS], weights, [0, 1, 2], lambda values: values == 2
    )
    row.update(dict(zip(MONTHS12_RATE_COLUMNS, months12_rates)))
    row.update(dict(zip(DAYS_ANY_RATE_COLUMNS, days_any_rates)))
    row.update(dict(zip(DAYS_2PLUS_RATE_COLUMNS, days_2plus_rates)))
    return row


def aggregate_panel(frame, group_columns, weight_column):
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else group_columns
    for keys, group in frame.groupby(grouper, observed=True, sort=True):
        keys = keys if isinstance(keys, tuple) else (keys,)
        row = dict(zip(group_columns, keys))
        row.update(summarise_group(group, weight_column))
        rows.append(row)
    return pd.DataFrame(rows)


### Result

All survey rates share the same missing-value and weighting rules. `effective_n` remains precision
metadata: it is not an activity variable and is not used to create the estimates themselves.


## 6. Annual panels (`wt_final`)

Annual estimates are produced separately for London, Inner/Outer London and the 32 boroughs.
Calendar-year external context and November–October weather are then attached at the matching
geographic level.


In [ ]:
london_annual = aggregate_panel(annual_respondents, ['year', 'survey_wave'], 'wt_final')
london_annual['geography_level'] = 'London'
london_annual['geography_code'] = 'LONDON_32'
london_annual['geography_name'] = 'London excluding City of London'

area_annual = aggregate_panel(
    annual_respondents, ['year', 'survey_wave', 'LondInOut', 'inner_outer'], 'wt_final'
)
area_annual['geography_level'] = 'InnerOuter'
area_annual['geography_code'] = area_annual['LondInOut'].map(AREA_CODES)
area_annual['geography_name'] = area_annual['inner_outer']

borough_annual = aggregate_panel(
    annual_respondents,
    ['year', 'survey_wave', 'gss_code', 'borough', 'inner_outer'],
    'wt_final',
)
borough_annual['geography_level'] = 'Borough'
borough_annual['geography_code'] = borough_annual['gss_code']
borough_annual['geography_name'] = borough_annual['borough']


### Attach annual context

Borough-year context is aggregated consistently to Inner/Outer and London before matching the survey-wave end year.


In [ ]:
# Attach Inner/Outer labels to the external borough-year source, then aggregate context.
context_geography = borough_annual[
    ['geography_code', 'geography_name', 'inner_outer']
].drop_duplicates()
borough_context_geo = borough_external_context.merge(
    context_geography[['geography_code', 'inner_outer']],
    on='geography_code',
    validate='many_to_one',
)
london_context_by_year = aggregate_borough_context(borough_context_geo, ['calendar_year'])
area_context_by_year = aggregate_borough_context(
    borough_context_geo, ['calendar_year', 'inner_outer']
)


def attach_annual_context(panel, level):
    result = panel.copy()
    result['context_reference_year'] = 2015 + result['year']
    if level == 'borough':
        context = borough_context_geo[
            ['calendar_year', 'geography_code']
            + SOCIODEMOGRAPHIC_CONTEXT_COLUMNS
            + EXTERNAL_QUALITY_COLUMNS
        ]
        result = result.merge(
            context,
            left_on=['context_reference_year', 'geography_code'],
            right_on=['calendar_year', 'geography_code'],
            validate='one_to_one',
        ).drop(columns='calendar_year')
    elif level == 'area':
        result = result.merge(
            area_context_by_year,
            left_on=['context_reference_year', 'inner_outer'],
            right_on=['calendar_year', 'inner_outer'],
            validate='one_to_one',
        ).drop(columns='calendar_year')
    else:
        result = result.merge(
            london_context_by_year,
            left_on='context_reference_year',
            right_on='calendar_year',
            validate='one_to_one',
        ).drop(columns='calendar_year')
    return result.merge(weather_annual_context, on='year', validate='many_to_one')


london_annual = attach_annual_context(london_annual, 'london')
area_annual = attach_annual_context(area_annual, 'area')
borough_annual = attach_annual_context(borough_annual, 'borough')


### Final annual schema

Identifiers, quality metadata, outcomes, activity fields and external context are placed in one stable order.


In [ ]:
ANNUAL_ORDER = ['year', 'survey_wave', 'geography_level', 'geography_code', 'geography_name']
MEASURE_ORDER = (
    OUTCOME_COLUMNS
    + COMPOSITION_FEATURES
    + QUALITY_COLUMNS
    + EXTERNAL_PANEL_COLUMNS
    + ACTIVITY_FEATURES
)
london_annual = london_annual[ANNUAL_ORDER + MEASURE_ORDER]
area_annual = area_annual[ANNUAL_ORDER + ['inner_outer'] + MEASURE_ORDER]
borough_annual = borough_annual[ANNUAL_ORDER + ['inner_outer'] + MEASURE_ORDER]
display(
    london_annual[
        [
            'survey_wave',
            'active_rate',
            'population_age_65_plus_rate',
            'economic_inactivity_rate',
            'earnings_relative_to_london',
            'net_migration_per_1000',
            'weather_mean_temperature_c',
        ]
    ]
)


### Findings

The annual outputs contain **8 London rows, 16 Inner/Outer rows and 256 borough rows**, all with
complete outcomes. London active share moves from **64.6% in 2015/16 to 66.4% in 2022/23**; the
65+ adult population share rises from 14.1% to 14.9%, motivating the later age decomposition.


## 7. Monthly panels (`wt_time`)

A balanced 96-month grid is retained at every geography level. Calendar-year socioeconomic
values are repeated only as source context; Notebook 02 applies the required 12-month lag before
forecasting.


In [ ]:
wave_labels = {wave['wave_index']: wave['survey_wave'] for wave in waves}
time_reference = pd.DataFrame({'month_index': range(1, 97)})
time_reference['year'] = ((time_reference['month_index'] - 1) // 12 + 1).astype(int)
time_reference['survey_wave'] = time_reference['year'].map(wave_labels)
time_reference['month_of_wave'] = ((time_reference['month_index'] - 1) % 12 + 1).astype(int)
time_reference['period_start'] = [str(FIRST_SURVEY_PERIOD + offset) for offset in range(96)]
time_reference['context_reference_year'] = pd.to_datetime(time_reference['period_start']).dt.year

borough_reference = (
    monthly_respondents[['LA_2023', 'gss_code', 'borough', 'LondInOut', 'inner_outer']]
    .drop_duplicates()
    .sort_values('borough')
    .reset_index(drop=True)
)
assert len(borough_reference) == 32


### Build balanced survey panels

London, Inner/Outer and borough estimates are merged onto complete time/geography grids.


In [ ]:
london_estimates = aggregate_panel(monthly_respondents, ['month_index'], 'wt_time')
london_monthly = time_reference.merge(
    london_estimates, on='month_index', how='left', validate='one_to_one'
)
london_monthly['geography_level'] = 'London'
london_monthly['geography_code'] = 'LONDON_32'
london_monthly['geography_name'] = 'London excluding City of London'


### Attach monthly weather

The common London weather series is joined only after the geographic survey estimates and borough-year context are complete.


In [ ]:
london_monthly = london_monthly.merge(
    london_context_by_year,
    left_on='context_reference_year',
    right_on='calendar_year',
    validate='many_to_one',
).drop(columns='calendar_year')

area_reference = pd.DataFrame(
    {
        'LondInOut': [1, 2],
        'inner_outer': [AREA_LABELS[1], AREA_LABELS[2]],
        'geography_code': [AREA_CODES[1], AREA_CODES[2]],
    }
)
area_estimates = aggregate_panel(monthly_respondents, ['month_index', 'LondInOut'], 'wt_time')
area_monthly = time_reference.merge(area_reference, how='cross').merge(
    area_estimates, on=['month_index', 'LondInOut'], how='left', validate='one_to_one'
)
area_monthly['geography_level'] = 'InnerOuter'
area_monthly['geography_name'] = area_monthly['inner_outer']
area_monthly = area_monthly.merge(
    area_context_by_year,
    left_on=['context_reference_year', 'inner_outer'],
    right_on=['calendar_year', 'inner_outer'],
    validate='many_to_one',
).drop(columns='calendar_year')

borough_estimates = aggregate_panel(monthly_respondents, ['month_index', 'LA_2023'], 'wt_time')
borough_monthly = time_reference.merge(borough_reference, how='cross').merge(
    borough_estimates, on=['month_index', 'LA_2023'], how='left', validate='one_to_one'
)
borough_monthly['geography_level'] = 'Borough'
borough_monthly['geography_code'] = borough_monthly['gss_code']
borough_monthly['geography_name'] = borough_monthly['borough']
borough_monthly = borough_monthly.merge(
    borough_context_geo[
        ['calendar_year', 'geography_code']
        + SOCIODEMOGRAPHIC_CONTEXT_COLUMNS
        + EXTERNAL_QUALITY_COLUMNS
    ],
    left_on=['context_reference_year', 'geography_code'],
    right_on=['calendar_year', 'geography_code'],
    validate='many_to_one',
).drop(columns='calendar_year')

for frame in [london_monthly, area_monthly, borough_monthly]:
    frame = frame  # retain object identity while merging weather below
london_monthly = london_monthly.merge(
    weather_monthly_context, on='period_start', validate='many_to_one'
)
area_monthly = area_monthly.merge(
    weather_monthly_context, on='period_start', validate='many_to_one'
)
borough_monthly = borough_monthly.merge(
    weather_monthly_context, on='period_start', validate='many_to_one'
)

TIME_ORDER = ['month_index', 'year', 'survey_wave', 'month_of_wave', 'period_start']
GEO_ORDER = ['geography_level', 'geography_code', 'geography_name']


### Final monthly schema

All three monthly levels use the same time, geography and measurement order.


In [ ]:
london_monthly = london_monthly[TIME_ORDER + GEO_ORDER + MEASURE_ORDER]
area_monthly = area_monthly[TIME_ORDER + GEO_ORDER + ['inner_outer'] + MEASURE_ORDER]
borough_monthly = borough_monthly[TIME_ORDER + GEO_ORDER + ['inner_outer'] + MEASURE_ORDER]
display(
    london_monthly[
        [
            'period_start',
            'active_rate',
            'weather_mean_temperature_c',
            'weather_total_precipitation_mm',
            'population_age_65_plus_rate',
        ]
    ].head()
)


### Findings

The monthly outputs contain **96 London rows, 192 Inner/Outer rows and 3,072 borough-month rows**.
There is one known empty survey estimate—Barking and Dagenham, October 2020—which remains `NaN`;
its population and weather context remain available.


## 8. Validation, documentation and export

The final stage checks keys, row counts, geography/time coverage, finite values, rate ranges,
positive denominators, coherent outcome shares, complete external fields and the single expected
empty survey cell. It then exports the six panels and a variable dictionary.


In [ ]:
def validate_panel(
    panel, keys, expected_rows, expected_geographies, expected_months=None, missing_rows=0
):
    assert len(panel) == expected_rows
    assert not panel.duplicated(keys).any()
    assert panel['geography_name'].nunique() == expected_geographies
    if expected_months is not None:
        assert set(panel['month_index']) == set(expected_months)
    observed_mask = panel[OUTCOME_COLUMNS].notna().all(axis=1)
    assert int((~observed_mask).sum()) == missing_rows
    observed = panel.loc[observed_mask]
    assert observed[OUTCOME_COLUMNS + COMPOSITION_FEATURES].notna().all().all()
    assert (
        observed[OUTCOME_COLUMNS + COMPOSITION_FEATURES]
        .apply(lambda x: x.between(0, 1))
        .all()
        .all()
    )
    assert observed['effective_n'].notna().all() and observed['effective_n'].gt(0).all()
    activity_values = observed[ACTIVITY_FEATURES].stack()
    assert activity_values.between(0, 1).all()
    assert observed[ACTIVITY_FEATURES].notna().any().all()
    assert np.allclose(observed[OUTCOME_COLUMNS].sum(axis=1), 1.0, atol=1e-10)

    assert panel[EXTERNAL_PANEL_COLUMNS].notna().all().all()
    assert np.isfinite(panel[EXTERNAL_PANEL_COLUMNS].to_numpy(float)).all()
    rate_columns = [
        'population_age_16_34_rate',
        'population_age_65_plus_rate',
        'population_female_rate',
        'economic_inactivity_rate',
        'weather_wet_day_rate',
    ]
    assert panel[rate_columns].apply(lambda x: x.between(0, 1)).all().all()
    assert panel['adult_population'].gt(0).all()
    assert panel['median_weekly_earnings_gbp'].gt(0).all()
    assert panel['earnings_relative_to_london'].gt(0).all()
    assert panel['weather_total_precipitation_mm'].ge(0).all()
    assert panel['weather_total_sunshine_hours'].ge(0).all()


### Execute panel assertions

A failed invariant stops the notebook before any file is written.


In [ ]:
validate_panel(london_annual, ['year'], 8, 1)
validate_panel(area_annual, ['year', 'geography_code'], 16, 2)
validate_panel(borough_annual, ['year', 'geography_code'], 256, 32)
validate_panel(london_monthly, ['month_index'], 96, 1, range(1, 97))
validate_panel(area_monthly, ['month_index', 'geography_code'], 192, 2, range(1, 97))
validate_panel(borough_monthly, ['month_index', 'geography_code'], 3072, 32, range(1, 97), 1)
assert 'City of London' not in set(borough_annual['geography_name'])
assert 'City of London' not in set(borough_monthly['geography_name'])


### Export validated panels

Only panels that pass every assertion reach the output directory.


In [ ]:
panels = {
    'london_annual': london_annual,
    'area_annual': area_annual,
    'borough_annual': borough_annual,
    'london_monthly': london_monthly,
    'area_monthly': area_monthly,
    'borough_monthly': borough_monthly,
}
for name, panel in panels.items():
    panel.to_csv(OUTPUTS[name], index=False, encoding='utf-8-sig')


### Build the variable dictionary

Every column is labelled by analytical role, source and unit.


In [ ]:
dictionary_rows = []
for column in borough_monthly.columns:
    if column in OUTCOME_COLUMNS:
        role, source, units = 'outcome', 'Active Lives / survey-weighted', 'proportion'
    elif column in COMPOSITION_FEATURES:
        role, source, units = (
            'survey composition or measurement control',
            'Active Lives / survey-weighted',
            'proportion',
        )
    elif column in QUALITY_COLUMNS or column in EXTERNAL_QUALITY_COLUMNS:
        role, source, units = (
            'quality metadata',
            'survey or external source',
            'effective count or percentage points',
        )
    elif column in SOCIODEMOGRAPHIC_CONTEXT_COLUMNS:
        role, source, units = (
            'external socioeconomic/demographic context',
            'ONS/ASHE/APS',
            'count, proportion, GBP or rate',
        )
    elif column in WEATHER_CONTEXT_COLUMNS:
        role, source, units = (
            'external weather context',
            'Open-Meteo historical weather data',
            'weather-specific',
        )
    elif column.startswith(('MONTHS12_RATE_', 'DAYS_ANY_RATE_', 'DAYS_2PLUS_RATE_')):
        role, source, units = 'activity predictor', 'Active Lives / survey-weighted', 'proportion'
    else:
        role, source, units = 'identifier', 'constructed or source geography/time', 'identifier'
    dictionary_rows.append({'variable': column, 'role': role, 'source': source, 'units': units})
variable_dictionary = pd.DataFrame(dictionary_rows)
variable_dictionary.to_csv(VARIABLE_DICTIONARY_OUTPUT, index=False, encoding='utf-8-sig')


### Compact delivery audit

The table below is the final shape and missing-cell check for downstream notebooks.


In [ ]:
summary = pd.DataFrame(
    [
        {'file': OUTPUTS[name].name, 'rows': len(panel), 'columns': len(panel.columns)}
        for name, panel in panels.items()
    ]
)
display(summary)
display(
    borough_monthly.loc[
        borough_monthly[OUTCOME_COLUMNS].isna().all(axis=1),
        TIME_ORDER + GEO_ORDER + ['adult_population', 'weather_mean_temperature_c'],
    ]
)


### Findings

All six panels pass their declared invariants. Each contains the same **372 activity variables**,
three survey-composition controls, eight socioeconomic/demographic fields, five weather fields and
explicit quality metadata. No model split or contemporaneous forecasting shortcut is embedded in
the prepared data.


## Modelling hand-off

The panels contain current-period context for description. Notebook 02 must create lags within
borough—12 months for monthly prediction and one survey year for annual prediction. `effective_n`
and confidence fields remain quality metadata, not predictors. Notebook 03 may use current-period
context for descriptive association and decomposition, but neither notebook estimates a policy or
causal effect.
